# Détection de fraude par carte de crédit

**Projet d'examen — Machine Learning — M2 Génie Informatique (Sujet A)**

**Problématique :** construire un système capable de détecter les transactions frauduleuses en minimisant à la fois les fraudes non détectées (coût direct) et les fausses alertes (coût opérationnel et expérience client).

**Jeu de données :** [Credit Card Fraud Detection](https://www.kaggle.com/mlg-ulb/creditcardfraud) — 284 807 transactions, 492 frauduleuses (0,17 %). Variables `V1`–`V28` (composantes ACP anonymisées), `Time`, `Amount`, cible `Class` (0 = légitime, 1 = fraude).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, average_precision_score,
    roc_curve, roc_auc_score, PrecisionRecallDisplay, RocCurveDisplay,
)
from sklearn.inspection import permutation_importance

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 1. Analyse exploratoire des données (EDA)

In [ ]:
df = pd.read_csv("../data/raw/creditcard.csv")
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe().T

In [ ]:
print("Valeurs manquantes par colonne :")
print(df.isna().sum().sum(), "valeurs manquantes au total")
print("\nDoublons :", df.duplicated().sum())

In [ ]:
class_counts = df["Class"].value_counts()
class_pct = df["Class"].value_counts(normalize=True) * 100
print(class_counts)
print(class_pct)

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, ax=ax)
ax.set_xticklabels(["Légitime (0)", "Fraude (1)"])
ax.set_ylabel("Nombre de transactions")
ax.set_title("Déséquilibre extrême des classes")
plt.tight_layout()
plt.savefig("../reports/figures/class_balance.png", dpi=150)
plt.show()

**Remarque méthodologique :** avec 0,17 % de fraudes, un modèle qui prédit systématiquement "légitime" atteindrait 99,83 % d'exactitude tout en étant totalement inutile. L'accuracy est donc un indicateur trompeur ici — voir section 5 pour les métriques réellement adaptées (précision, rappel, F1, AUC-PR).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["Amount"], bins=50, ax=axes[0])
axes[0].set_title("Distribution de Amount")
sns.histplot(df["Time"], bins=50, ax=axes[1])
axes[1].set_title("Distribution de Time (secondes depuis la 1ère transaction)")
plt.tight_layout()
plt.savefig("../reports/figures/amount_time_distributions.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(x="Class", y="Amount", data=df[df["Amount"] < df["Amount"].quantile(0.99)], ax=ax)
ax.set_xticklabels(["Légitime", "Fraude"])
ax.set_title("Montant des transactions selon la classe (hors extrêmes)")
plt.tight_layout()
plt.savefig("../reports/figures/amount_by_class.png", dpi=150)
plt.show()

In [ ]:
corr = df.corr(numeric_only=True)["Class"].sort_values(key=abs, ascending=False)
print(corr.head(15))

fig, ax = plt.subplots(figsize=(6, 8))
corr.drop("Class").plot(kind="barh", ax=ax)
ax.set_title("Corrélation de chaque variable avec Class")
plt.tight_layout()
plt.savefig("../reports/figures/correlation_with_class.png", dpi=150)
plt.show()

## 2. Prétraitement et découpage train/test

Le découpage est effectué **avant** toute transformation (standardisation, sur-échantillonnage) afin d'éviter toute fuite de données. Le jeu de test est mis de côté et n'intervient plus jusqu'à l'évaluation finale (section 5).

In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "— fraudes:", y_train.sum(), f"({y_train.mean()*100:.3f}%)")
print("Test: ", X_test.shape, "— fraudes:", y_test.sum(), f"({y_test.mean()*100:.3f}%)")

`V1`–`V28` sont déjà issues d'une ACP (donc déjà centrées/réduites en amont par les auteurs du dataset). Seules `Time` et `Amount` nécessitent une standardisation, apprise uniquement sur le train.

In [ ]:
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[["Time", "Amount"]] = scaler.fit_transform(X_train[["Time", "Amount"]])
X_test_scaled[["Time", "Amount"]] = scaler.transform(X_test[["Time", "Amount"]])

## 3. Modélisation

Trois familles d'algorithmes comparées sur un protocole identique :
1. **Régression logistique** (`class_weight="balanced"`) — modèle linéaire, interprétable, sert de référence.
2. **Random Forest** — modèle d'ensemble à base d'arbres, capture les non-linéarités et interactions.
3. **k plus proches voisins (k-NN)** — modèle non paramétrique, point de comparaison ; on s'attend à une performance plus faible et/ou un coût de calcul élevé en grande dimension (28 variables + Time/Amount), ce qui sera discuté en section 7.

In [ ]:
models = {
    "Logistic Regression": ImbPipeline([
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Random Forest": ImbPipeline([
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    "k-NN": ImbPipeline([
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
    ]),
}

## 4. Sélection de modèle et recherche d'hyperparamètres

Validation croisée **stratifiée** (pour préserver le taux de fraude dans chaque pli) avec `RandomizedSearchCV`. SMOTE est intégré dans le pipeline `imblearn` afin d'être ré-appliqué uniquement sur les plis d'entraînement à chaque itération de la validation croisée — évitant toute fuite d'information vers les plis de validation.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

param_distributions = {
    "Logistic Regression": {"clf__C": [0.01, 0.1, 1, 10, 100]},
    "Random Forest": {
        "clf__n_estimators": [100, 200, 300],
        "clf__max_depth": [None, 10, 20, 30],
        "clf__min_samples_leaf": [1, 2, 5],
    },
    "k-NN": {"clf__n_neighbors": [3, 5, 7, 11]},
}

best_estimators = {}
cv_results = {}

for name, pipeline in models.items():
    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_distributions[name],
        n_iter=8,
        scoring="average_precision",  # AUC-PR : adapté au déséquilibre extrême
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
    )
    search.fit(X_train_scaled, y_train)
    best_estimators[name] = search.best_estimator_
    cv_results[name] = search
    print(f"{name}: best AUC-PR (CV) = {search.best_score_:.4f}, best params = {search.best_params_}")

## 5. Évaluation rigoureuse

Évaluation finale sur le jeu de test tenu à l'écart de tout le processus de sélection de modèle. Métriques : précision, rappel, F1, AUC-PR (prioritaire ici sur l'AUC-ROC, trompeuse en cas de déséquilibre extrême), matrice de confusion.

In [ ]:
results_summary = []

fig_pr, ax_pr = plt.subplots(figsize=(6, 5))
fig_roc, ax_roc = plt.subplots(figsize=(6, 5))

for name, estimator in best_estimators.items():
    y_pred = estimator.predict(X_test_scaled)
    y_proba = estimator.predict_proba(X_test_scaled)[:, 1]

    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, digits=4))

    ap = average_precision_score(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)
    results_summary.append({"model": name, "AUC-PR": ap, "AUC-ROC": roc_auc})

    PrecisionRecallDisplay.from_predictions(y_test, y_proba, name=name, ax=ax_pr)
    RocCurveDisplay.from_predictions(y_test, y_proba, name=name, ax=ax_roc)

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Légitime", "Fraude"])
    disp.plot(cmap="Blues")
    plt.title(f"Matrice de confusion — {name}")
    plt.savefig(f"../reports/figures/confusion_matrix_{name.replace(' ', '_')}.png", dpi=150)
    plt.show()

ax_pr.set_title("Courbes Précision-Rappel")
ax_roc.set_title("Courbes ROC")
fig_pr.savefig("../reports/figures/pr_curves.png", dpi=150)
fig_roc.savefig("../reports/figures/roc_curves.png", dpi=150)
plt.show()

results_df = pd.DataFrame(results_summary).sort_values("AUC-PR", ascending=False)
results_df

### Stabilité des résultats (variance entre plis de validation croisée)

In [ ]:
for name, search in cv_results.items():
    scores = search.cv_results_["mean_test_score"][search.best_index_]
    std = search.cv_results_["std_test_score"][search.best_index_]
    print(f"{name}: AUC-PR = {scores:.4f} ± {std:.4f} (écart-type entre plis)")

## 6. Interprétabilité

Importance des variables pour le meilleur modèle (à sélectionner selon les résultats de la section 5 — probablement Random Forest).

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = best_estimators[best_model_name]
print("Meilleur modèle retenu :", best_model_name)

In [ ]:
# Feature importance native (si modèle à base d'arbres)
clf = best_model.named_steps["clf"]
if hasattr(clf, "feature_importances_"):
    importances = pd.Series(clf.feature_importances_, index=X_train_scaled.columns).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(6, 8))
    importances.head(15).plot(kind="barh", ax=ax)
    ax.invert_yaxis()
    ax.set_title(f"Feature importance native — {best_model_name}")
    plt.tight_layout()
    plt.savefig("../reports/figures/feature_importance.png", dpi=150)
    plt.show()
else:
    print("Le modèle retenu n'expose pas de feature_importances_ natives — utiliser la permutation importance ci-dessous.")

In [ ]:
# Permutation importance (agnostique au modèle) — échantillon pour limiter le temps de calcul
sample_idx = X_test_scaled.sample(n=5000, random_state=RANDOM_STATE).index
perm = permutation_importance(
    best_model, X_test_scaled.loc[sample_idx], y_test.loc[sample_idx],
    n_repeats=10, random_state=RANDOM_STATE, scoring="average_precision", n_jobs=-1,
)
perm_importances = pd.Series(perm.importances_mean, index=X_train_scaled.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(6, 8))
perm_importances.head(15).plot(kind="barh", ax=ax)
ax.invert_yaxis()
ax.set_title(f"Permutation importance (AUC-PR) — {best_model_name}")
plt.tight_layout()
plt.savefig("../reports/figures/permutation_importance.png", dpi=150)
plt.show()

*(Optionnel — valeurs de Shapley via SHAP, à ajouter si le temps le permet, notamment pour illustrer des cas individuels de transactions frauduleuses détectées/manquées.)*

## 7. Analyse critique et discussion

*(À rédiger à partir des résultats réels obtenus ci-dessus — ne pas généraliser sans ancrage dans les chiffres du projet.)*

**Limites du meilleur modèle et cas d'erreur typiques**
- ...

**Compromis précision/rappel du point de vue métier**
- Coût d'une fraude manquée (faux négatif) vs. coût d'une transaction légitime bloquée (faux positif) : ...
- Seuil de décision retenu et justification : ...

**Biais potentiels du jeu de données**
- Transactions européennes de septembre 2013 uniquement : représentativité limitée dans le temps et l'espace (pas de fraude « moderne » type carding automatisé, pas de diversité géographique).
- Anonymisation par ACP (`V1`–`V28`) : empêche toute vérification de biais liés à des variables métier explicites (pays, type de commerçant, etc.).
- Conséquences d'un déploiement en production sur des données actuelles : ...

**Pistes d'amélioration concrètes**
- ...